# Setup Folders

In [ ]:
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists("src"):
        !git clone https://github.com/HenriqueSchmitz/world-models-implementation temp_repo
        !mv temp_repo/* .
        !rm -rf temp_repo
        !sudo apt-get install -y swig
        %pip install -q -r requirements.txt

In [ ]:
import os
from src.utils.colab import is_environment_colab_notebook
from src.utils.secrets import get_secret

remote_folder = None
if is_environment_colab_notebook():
    remote_folder = get_secret("worldModelsFolderPath")
    data_path = os.path.join(remote_folder, "data.tar")
    !cp "$data_path" .
    drive.flush_and_unmount()
    !tar -xf data.tar
    !rm -rf data.tar
    drive.mount('/content/drive')

settings_path = "./settings.json"
data_folder = "./data"
vae_folder = os.path.join(remote_folder, "weights/vae") if is_environment_colab_notebook() else "./weights/vae"
vae_folder = vae_folder if os.path.exists(vae_folder) else  "./weights/vae"
output_folder = os.path.join(remote_folder, "weights/worldmodel") if is_environment_colab_notebook() else "./weights/worldmodel"
os.makedirs(output_folder, exist_ok=True)

local_data_folder = None

# Training Settings

In [ ]:
from src.utils.logging import get_logger
LOG_LEVEL = "INFO"
logger = get_logger(LOG_LEVEL)

In [ ]:
import json

with open(settings_path, "r") as settings_file:
    settings = json.load(settings_file)
    IMAGE_CHANNELS = settings["vae"]["model"]["image_channels"]
    OBSERVATION_DIM = settings["vae"]["model"]["observation_dim"]
    VAE_HIDDEN_DIM = settings["vae"]["model"]["hidden_dim"]
    REPRESENTATION_DIM = settings["vae"]["model"]["representation_dim"]
    INPUT_STATE_DIM = settings["world_model"]["model"]["input_state_dim"]
    OUTPUT_STATE_DIM = settings["world_model"]["model"]["output_state_dim"]
    HIDDEN_DIM = settings["world_model"]["model"]["hidden_dim"]
    NUM_GAUSSIANS = settings["world_model"]["model"]["num_gaussians"]
    MIN_SIGMA = settings["world_model"]["model"]["min_sigma"]
    MAX_SIGMA = settings["world_model"]["model"]["max_sigma"]
    RNN_INPUT_DIM = REPRESENTATION_DIM + INPUT_STATE_DIM
    RNN_OUTPUT_DIM = REPRESENTATION_DIM + OUTPUT_STATE_DIM

    EPOCHS = settings["world_model"]["train"]["epochs"]
    LR = settings["world_model"]["train"]["learning_rate"]
    MAX_NORM = settings["world_model"]["train"]["max_norm"]
    TRAIN_RATIO = settings["world_model"]["train"]["train_ratio"]
    EPOCHS_BETWEEN_TESTS = settings["world_model"]["train"]["epochs_between_tests"]
    BATCH_SIZE = settings["world_model"]["train"]["batch_size"]
    SEQ_LEN = settings["world_model"]["train"]["sequence_length"]
    NUM_PRELOAD_FILES = settings["world_model"]["train"]["num_preload_files"]
    NUM_DATASET_WORKERS = settings["world_model"]["train"]["num_dataset_workers"]
    EARLY_STOPPING_TOLERANCE = settings["world_model"]["train"]["early_stopping_tolerance"]
    EARLY_STOPPING_MIN_DELTA = settings["world_model"]["train"]["early_stopping_min_delta"]

WANDB_PROJECT = "world-models-paper"
WANDB_RUN_NAME = "world_model"

def log_settings():
    logger.info(f"IMAGE_CHANNELS: {IMAGE_CHANNELS}")
    logger.info(f"OBSERVATION_DIM: {OBSERVATION_DIM}")
    logger.info(f"VAE_HIDDEN_DIM: {VAE_HIDDEN_DIM}")
    logger.info(f"REPRESENTATION_DIM: {REPRESENTATION_DIM}")
    logger.info(f"INPUT_STATE_DIM: {INPUT_STATE_DIM}")
    logger.info(f"OUTPUT_STATE_DIM: {OUTPUT_STATE_DIM}")
    logger.info(f"HIDDEN_DIM: {HIDDEN_DIM}")
    logger.info(f"NUM_GAUSSIANS: {NUM_GAUSSIANS}")
    logger.info(f"MIN_SIGMA: {MIN_SIGMA}")
    logger.info(f"MAX_SIGMA: {MAX_SIGMA}")
    logger.info(f"RNN_INPUT_DIM: {RNN_INPUT_DIM}")
    logger.info(f"RNN_OUTPUT_DIM: {RNN_OUTPUT_DIM}")
    logger.info("========================================")
    logger.info(f"EPOCHS: {EPOCHS}")
    logger.info(f"LR: {LR}")
    logger.info(f"TRAIN_RATIO: {TRAIN_RATIO}")
    logger.info(f"EPOCHS_BETWEEN_TESTS: {EPOCHS_BETWEEN_TESTS}")
    logger.info(f"BATCH_SIZE: {BATCH_SIZE}")
    logger.info(f"SEQ_LEN: {SEQ_LEN}")
    logger.info(f"NUM_PRELOAD_FILES: {NUM_PRELOAD_FILES}")
    logger.info(f"NUM_DATASET_WORKERS: {NUM_DATASET_WORKERS}")
    logger.info(f"EARLY_STOPPING_TOLERANCE: {EARLY_STOPPING_TOLERANCE}")
    logger.info(f"EARLY_STOPPING_MIN_DELTA: {EARLY_STOPPING_MIN_DELTA}")
    logger.info("========================================")
    logger.info(f"WANDB_PROJECT: {WANDB_PROJECT}")
    logger.info(f"WANDB_RUN_NAME: {WANDB_RUN_NAME}")

log_settings()

# Setup

In [ ]:
import torch
from torch.utils.data import DataLoader

from src.datasets.simulation_steps_dataset import SimulationStepsDataset
from src.models.vae import ConvVAE
from src.models.worldmodel import MdnRnn
from src.training.early_stopping import EarlyStopping
from src.training.worldmodel_trainer import WorldModelTrainer
from src.utils.torch import get_device
from src.utils.secrets import get_secret

In [ ]:
DEVICE = get_device()

# Load Dataset

In [ ]:
train_dataset, test_dataset = SimulationStepsDataset.train_test_split(data_folder,
                                                                      local_data_folder=local_data_folder,
                                                                      num_preloaded_files=NUM_PRELOAD_FILES,
                                                                      num_workers=NUM_DATASET_WORKERS,
                                                                      train_ratio=TRAIN_RATIO,
                                                                      shuffle_files=True,
                                                                      shuffle_file_samples=True, # Can only be done since sequences are built before shuffle
                                                                      kwargs={"sequence_length": SEQ_LEN})

In [ ]:
train_size = len(train_dataset)
test_size = len(test_dataset)
logger.info(f"Train: {train_size}")
logger.info(f"Test: {test_size}")

In [ ]:
example = next(train_dataset)
example_observation = example[0][0].unflatten(0, (IMAGE_CHANNELS, OBSERVATION_DIM, OBSERVATION_DIM))
example_input_state = example[0][1]
example_output_state = example[0][2]
logger.info(example_observation.shape)
logger.info(example_input_state.shape)
logger.info(example_output_state.shape)

In [ ]:
def custom_collate_fn(batch):
    observations = torch.stack([item[0] for item in batch])
    input_states = torch.stack([item[1] for item in batch])
    output_states = torch.stack([item[2] for item in batch])
    return observations, input_states, output_states

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=custom_collate_fn)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=custom_collate_fn)

In [ ]:
train_batches = len(train_dataloader)
test_batches = len(test_dataloader)
logger.info(f"Train batches: {train_batches}")
logger.info(f"Test batches: {test_batches}")

# Train

In [ ]:
vae_path = os.path.join(vae_folder, f"model.pth")
vae = ConvVAE(image_channels=IMAGE_CHANNELS,
              h_dim=VAE_HIDDEN_DIM,
              z_dim=REPRESENTATION_DIM,
              device=DEVICE,
              weights_path=vae_path)
vae.freeze_weights().eval() # We want to train only the world model

In [ ]:
model = MdnRnn(input_size=RNN_INPUT_DIM,
               hidden_size=HIDDEN_DIM,
               output_size=RNN_OUTPUT_DIM,
               num_gaussians=NUM_GAUSSIANS,
               min_sigma=MIN_SIGMA,
               max_sigma=MAX_SIGMA,
               device=DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
early_stopping = EarlyStopping(tolerance=EARLY_STOPPING_TOLERANCE, min_delta=EARLY_STOPPING_MIN_DELTA)

In [ ]:
try:
    wandb_setup = {
        "api_key": get_secret('wandbApiKey'),
        "project": WANDB_PROJECT,
        "run_name": WANDB_RUN_NAME,
        "config": {
            "epochs": EPOCHS,
            "batch_size_loader": BATCH_SIZE,
            "learning_rate": LR,
            "train_ratio": TRAIN_RATIO,
            "hidden_dim": HIDDEN_DIM,
            "representation_dim": REPRESENTATION_DIM,
            "architecture": "MDN-RNN",
            "train_dataset_size": train_size,
            "test_dataset_size": test_size,
            "train_batches": train_batches,
            "test_batches": test_batches,
            "preload_files": NUM_PRELOAD_FILES,
            "num_dataset_workers": NUM_DATASET_WORKERS
        }
    }
except Exception as e:
    logger.error(f"Ignoring wandb logging, could not retrieve wandbApiKey secret. Error: {str(e)}")
    wandb_setup = None

In [ ]:
trainer = WorldModelTrainer(model=model,
                            weights_folder=output_folder,
                            train_dataloader=train_dataloader,
                            optimizer=optimizer,
                            num_epochs=EPOCHS,
                            batch_size=BATCH_SIZE,
                            load_checkpoint=True,
                            max_norm=MAX_NORM,
                            device=DEVICE,
                            test_dataloader=test_dataloader,
                            epochs_between_tests=EPOCHS_BETWEEN_TESTS,
                            wandb_setup=wandb_setup,
                            logger=logger,
                            vae=vae,
                            observation_dim=OBSERVATION_DIM,
                            seq_len=SEQ_LEN)

In [ ]:
# Repeat logging of important information so it is available on wandb run
log_settings()
get_device(logger)

In [ ]:
trainer.train()